# 3D BraTS 2020 Brain Tumor Segmentation (T1 Modality Only) - Production PyTorch Pipeline — v3

**Problem Definition & Clinical Context:**
Gliomas are the most common primary brain tumors in adults, requiring precise segmentation of sub-regions for surgical planning and prognostic assessment. The multimodal Brain Tumor Segmentation (BraTS) benchmark traditionally provides four MRI modalities (T1, T1CE, T2, FLAIR).

**Project Scope & Strictly Enforced Modality Constraint:**
In clinical deployment, acquiring complete multimodal MRI suites can be time-consuming, artifact-prone, or unavailable in resource-constrained centers. This production pipeline investigates and implements an end-to-end deep learning segmentation model utilizing **EXCLUSIVELY Native T1-weighted (T1) MRI scans (`Patient_xxx_t1.nii`)** to predict the complete tumor sub-region mask (`Patient_xxx_seg.nii`).

**Tumor Sub-Region Mapping (BraTS 2020 Standard):**
* **Class 0:** Background / Healthy brain tissue.
* **Class 1:** Necrotic and Non-Enhancing Tumor Core (NCR/NET).
* **Class 2:** Peritumoral Edema (ED).
* **Class 3:** GD-Enhancing Tumor (ET) (Re-mapped from raw label 4 to class index 3 for contiguous one-hot encoding).

**Architectural & Engineering Highlights:**
* **Framework:** PyTorch 2.x with Automatic Mixed Precision (AMP).
* **Hardware Optimization:** Distributed multi-GPU execution specifically tuned for Kaggle Dual Tesla T4 GPUs.
* **Architecture:** Lightweight Attention U-Net with spatial attention gates, Mish activations, and Kaiming initialization.
* **Optimization:** Combined Binary Cross-Entropy + Soft Dice Loss with Cosine Annealing and Linear Warmup.
* **Evaluation:** Full quantitative suite computing multi-class Dice, Intersection over Union (IoU), Precision, Recall, F1-score, and 95th percentile Hausdorff Distance (HD95).

---

## v2 Changelog (fixes applied per `training_bottleneck_analysis.md`)

1. **`cudnn.benchmark` flipped `False -> True`** (Section 1). Safe for this pipeline since every input is a fixed 128x128 slice; re-enables cuDNN's convolution auto-tuner.
2. **Eliminated full-volume reloads on every `__getitem__`.** Added a one-time slice pre-caching step (new Section 6) that converts every needed axial slice to a small `float32` `.npy` file. The Dataset (Section 7) now does a trivial `np.load()` per item instead of `nib.load(...).get_fdata()` on the entire 155-slice volume. This was the single biggest bottleneck (10-50x DataLoader slowdown per the analysis).
3. **`get_fdata()` now always called with `dtype=np.float32`** everywhere a NIfTI volume is loaded (pre-caching, visualization, inference), instead of silently defaulting to `float64`.
4. **`NUM_WORKERS` from `PipelineConfig` is now actually wired into the DataLoaders.** It was defined but hardcoded to `0` in v1. Since slice I/O is now trivial, multiple workers genuinely overlap I/O with GPU compute instead of contending for slow full-volume disk reads.
5. **Removed redundant/duplicate dataset scanning.** v1 called `scan_and_verify_dataset()` independently in three separate cells. v2 consolidates this into a single canonical Section 3 cell (synthetic-data fallback preserved for non-Kaggle environments).
6. **Deterministic train/val/test split.** v1's canonical split cell seeded the shuffle correctly, but an earlier redundant cell shuffled with the unseeded global `random` state. Now there is exactly one seeded `random.Random(PipelineConfig.SEED)` shuffle.
7. **Removed leftover interactive debug cells** (`/kaggle/input` directory listings, ad-hoc `os.walk` search, scattered one-line `print()` sanity checks). Consolidated the useful ones into a single "Pre-Flight System Check" cell.

> [!NOTE]
> After the caching fix, training throughput should improve by **10-50x** on repeat epochs. The bottleneck shifts from CPU/disk to actual GPU compute. The first run will take extra time up front to build the `.npy` cache — this is a one-time cost (the pre-caching cell is idempotent and skips already-cached slices on subsequent runs).

---

## v3 Changelog (training-quality fixes, based on observed low Dice scores)

After running v2, per-class validation Dice showed a clear pattern: Enhancing Tumor (ET) started near 0 and stayed weak (~0.25–0.28), Edema (ED) was the strongest class (~0.45–0.47), and Necrotic Core (NCR) sat in between (~0.22–0.29). Mean Dice oscillated in the 0.31–0.34 range from epoch 9 onward without climbing further. Two concrete fixes are applied in v3:

1. **Loss rebalanced from Dice to Tversky, and reweighted toward the segmentation term.** `MultiClassTverskyLoss` replaces `MultiClassDiceLoss` inside `CombinedSegmentationLoss` (Section 9). Tversky's `alpha`/`beta` let false negatives be penalized more heavily than false positives (`alpha=0.7`, `beta=0.3` by default) — this directly targets small, easily-missed structures like ET and NCR, which plain Dice under-penalizes relative to the dominant background/ED classes. The CE/segmentation-loss weighting also shifted from 0.5/0.5 to 0.3/0.7 so the segmentation term (not per-pixel CE, which is dominated by the easy background class) drives more of the gradient.
2. **LR schedule switched from `CosineAnnealingLR` to `ReduceLROnPlateau`.** The v2 run showed val Dice oscillating epoch-to-epoch (0.3407 → 0.3367 → 0.3153 → 0.3214) while train loss kept falling smoothly — the classic signature of an LR that's still too high for this stage of training under a fixed cosine schedule. `ReduceLROnPlateau` now monitors val Dice directly and only decays the LR (`factor=0.5`, `patience=3`) once improvement actually stalls, instead of decaying on a fixed epoch-count schedule regardless of real progress.

A third item — **recalibrated expectations for T1-only training** — is not a code change. See the note added after Section 13 (Evaluation Benchmark): given that native T1 does not contain the contrast-enhancement signal that *defines* the Enhancing Tumor class, a mean Dice in the ~0.35–0.45 range is a plausible practical ceiling for this modality-restricted setup, not evidence of a broken pipeline.

---

## Section 1 — Environment Setup, Modular Imports, and Deterministic Seeding


In [ ]:
"""
Section 1: Environment Setup, Modular Imports, and Deterministic Seeding
-------------------------------------------------------------------------
All necessary libraries for medical NIfTI volume processing, tensor operations,
deep learning model definitions, multi-GPU scaling, and qualitative visualization.
"""

import os
import gc
import glob
import math
import shutil
import random
import warnings
from pathlib import Path
from typing import List, Tuple, Dict, Optional, Union

import cv2
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import nibabel as nib
from scipy import ndimage

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torch.utils.tensorboard import SummaryWriter

from tqdm.auto import tqdm

# Configure display formatting and suppress verbose warnings
warnings.filterwarnings("ignore", category=UserWarning)
np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")


def set_deterministic_seed(seed: int = 42) -> None:
    """
    Enforce complete reproducibility across Python, NumPy, and PyTorch CPU/GPU operations.

    Args:
        seed (int): The random seed value to initialize all RNG states.
    """
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Enforce deterministic cuDNN backend algorithms
        torch.backends.cudnn.deterministic = True
        # [v2 FIX] Was `False`, which disables cuDNN's convolution algorithm auto-tuner.
        # All inputs in this pipeline are fixed-size 128x128 slices, so the tuner can safely
        # pick the fastest algorithm once and reuse it -- 20-100% faster GPU ops.
        torch.backends.cudnn.benchmark = True

    print(f"[System] Global execution state locked to deterministic seed: {seed}")


set_deterministic_seed(42)



---

## Section 2 — Production Pipeline Configuration


In [ ]:
"""
Section 2: Production Pipeline Configuration
--------------------------------------------
Encapsulates all hyperparameters, hardware parameters, dataset paths, and slice bounds
into an immutable object-oriented dataclass structure.
"""

class PipelineConfig:
    # Project Identity
    PROJECT_NAME: str = "BraTS2020_T1_Segmentation_DualT4"
    MODALITY: str = "t1"           # Strictly enforce T1 native MRI modality only
    TARGET_SUFFIX: str = "seg"     # Ground truth segmentation mask suffix

    # Dataset Locations (Adapted for Kaggle Notebook structure)
    TRAIN_DATASET_DIR = "/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"
    VAL_DATASET_DIR: str = "/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation/BraTS2020_ValidationData/MICCAI_BraTS2020_ValidationData"
    CHECKPOINT_DIR: str = "./checkpoints"
    LOG_DIR: str = "./runs/t1_attention_unet"

    # [v2 NEW] Pre-cached per-slice .npy store (built once in Section 6, consumed by Section 7)
    CACHE_DIR: str = "./slice_cache"

    # Dimensional & Volume Extraction Parameters
    IMG_SIZE: int = 128            # Spatial resolution of resized 2D axial slices
    VOLUME_SLICES: int = 100       # Total axial slices extracted per 3D volume
    VOLUME_START_AT: int = 22      # Index of the initial axial slice (skipping empty cranial apex/base)
    NUM_CLASSES: int = 4           # 0: Background, 1: Necrotic/Core, 2: Edema, 3: Enhancing Tumor
    SEED = 42

    # Training & Optimization Hyperparameters
    BATCH_SIZE: int = 32           # High batch size optimized for Dual 16GB Tesla T4 GPUs
    EPOCHS: int = 30               # Full convergence training schedule
    LEARNING_RATE: float = 3e-4    # Maximum learning rate for AdamW optimizer
    WEIGHT_DECAY: float = 1e-4     # L2 regularization coefficient
    WARMUP_EPOCHS: int = 3         # Linear learning rate warmup iterations
    GRAD_CLIP_NORM: float = 1.0    # Maximum gradient norm threshold

    # [v3 NEW] Loss Balancing (CrossEntropy vs. Tversky segmentation term)
    CE_WEIGHT: float = 0.3         # Down-weighted -- per-pixel CE is dominated by the easy background class
    TVERSKY_WEIGHT: float = 0.7    # Up-weighted -- drives most of the gradient toward tumor sub-regions
    TVERSKY_ALPHA: float = 0.7     # False-negative penalty (catches small/missed structures like ET, NCR)
    TVERSKY_BETA: float = 0.3      # False-positive penalty

    # [v3 NEW] Plateau-Based LR Scheduling (replaces fixed CosineAnnealingLR)
    LR_PLATEAU_FACTOR: float = 0.5     # Multiply LR by this factor when val Dice plateaus
    LR_PLATEAU_PATIENCE: int = 3       # Epochs without val Dice improvement before decaying LR
    LR_MIN: float = 1e-6               # Floor for LR decay

    # Hardware & Multiprocessing Optimization
    NUM_WORKERS: int = 4           # Parallel CPU workers for DataLoader (now actually used -- see Section 7)
    PIN_MEMORY: bool = True        # Pin CPU host memory for rapid PCIe GPU transfer
    USE_AMP: bool = True           # Enable Automatic Mixed Precision (FP16)

    @classmethod
    def setup_directories(cls) -> None:
        """Create necessary output directories for logs, model weights, and slice cache."""
        os.makedirs(cls.CHECKPOINT_DIR, exist_ok=True)
        os.makedirs(cls.LOG_DIR, exist_ok=True)
        os.makedirs(cls.CACHE_DIR, exist_ok=True)

    @classmethod
    def get_device(cls) -> torch.device:
        """Detect available compute hardware and initialize GPU environment."""
        if torch.cuda.is_available():
            device_count = torch.cuda.device_count()
            print(f"[Hardware] Detected {device_count} CUDA GPU(s):")
            for i in range(device_count):
                print(f"  -> GPU {i}: {torch.cuda.get_device_name(i)}")
            return torch.device("cuda")
        else:
            print("[Hardware] WARNING: CUDA not detected. Falling back to CPU execution.")
            return torch.device("cpu")


PipelineConfig.setup_directories()
DEVICE = PipelineConfig.get_device()



---

## Section 3 — Exploratory Data Analysis, Sanity Verification & Data Splitting


In [ ]:
"""
Section 3: Exploratory Data Analysis, Sanity Verification & Data Splitting
--------------------------------------------------------------------------
Discovers patient volume directories, performs integrity checks against corrupted filenames,
validates modality availability, and constructs deterministic train/val/test splits.

[v2 FIX] v1 called `scan_and_verify_dataset()` independently across three separate cells
(each traversing all 369 patient folders). This is now the single canonical call.
[v2 FIX] The train/val/test shuffle is now deterministically seeded via
`random.Random(PipelineConfig.SEED)` -- a redundant v1 cell previously shuffled with the
unseeded global `random` module state, undermining reproducibility.
"""

def scan_and_verify_dataset(base_dir: str) -> List[str]:
    """
    Traverse dataset base directory, verify patient folders, and filter out known corrupted entries.

    Args:
        base_dir (str): Root directory path containing patient folders.

    Returns:
        List[str]: Clean, verified list of absolute patient directory paths.
    """
    if not os.path.exists(base_dir):
        print(f"[Sanity Check] Directory not found at {base_dir}. Using mock verification paths if local.")
        return []

    patient_dirs = [f.path for f in os.scandir(base_dir) if f.is_dir()]

    # Filter out known ill-formatted BraTS 2020 training case (BraTS20_Training_355)
    corrupt_cases = ["BraTS20_Training_355"]
    clean_dirs = []

    for p_dir in sorted(patient_dirs):
        folder_name = os.path.basename(p_dir)
        if any(corrupt in folder_name for corrupt in corrupt_cases):
            print(f"[Sanity Check] Excluding known corrupted volume case: {folder_name}")
            continue

        # Verify mandatory T1 scan and segmentation mask files exist
        t1_path = os.path.join(p_dir, f"{folder_name}_{PipelineConfig.MODALITY}.nii")
        seg_path = os.path.join(p_dir, f"{folder_name}_{PipelineConfig.TARGET_SUFFIX}.nii")

        if os.path.exists(t1_path) and os.path.exists(seg_path):
            clean_dirs.append(p_dir)
        else:
            print(f"[Sanity Check] Missing required T1/seg NIfTI files in: {folder_name}")

    return clean_dirs


# Scan dataset once (canonical call)
all_patient_dirs = scan_and_verify_dataset(PipelineConfig.TRAIN_DATASET_DIR)

if len(all_patient_dirs) == 0:
    print("[System] Kaggle dataset path absent. Creating synthetic in-memory patient directory structure for verification.")
    synthetic_base = "./synthetic_brats_data"
    os.makedirs(synthetic_base, exist_ok=True)
    all_patient_dirs = []

    for idx in range(1, 21):
        p_name = f"BraTS20_Training_{idx:03d}"
        p_dir = os.path.join(synthetic_base, p_name)
        os.makedirs(p_dir, exist_ok=True)

        # Create synthetic 3D volume (155 slices, 240x240)
        synth_t1 = nib.Nifti1Image(np.random.rand(240, 240, 155).astype(np.float32), np.eye(4))
        synth_seg = nib.Nifti1Image(np.random.choice([0, 1, 2, 4], size=(240, 240, 155), p=[0.95, 0.02, 0.02, 0.01]).astype(np.uint8), np.eye(4))

        nib.save(synth_t1, os.path.join(p_dir, f"{p_name}_t1.nii"))
        nib.save(synth_seg, os.path.join(p_dir, f"{p_name}_seg.nii"))
        all_patient_dirs.append(p_dir)

if len(all_patient_dirs) == 0:
    raise FileNotFoundError(
        f"No valid BraTS2020 cases found in\n{PipelineConfig.TRAIN_DATASET_DIR}"
    )

# [v2 FIX] Deterministic, seeded shuffle -- single source of truth for the split.
rng = random.Random(PipelineConfig.SEED)
rng.shuffle(all_patient_dirs)

num_total = len(all_patient_dirs)
train_end = int(num_total * 0.70)
val_end = int(num_total * 0.85)

train_patient_dirs = all_patient_dirs[:train_end]
val_patient_dirs = all_patient_dirs[train_end:val_end]
test_patient_dirs = all_patient_dirs[val_end:]

print("=" * 60)
print(f"Validated patients : {num_total}")
print(f"Training           : {len(train_patient_dirs)}")
print(f"Validation         : {len(val_patient_dirs)}")
print(f"Testing            : {len(test_patient_dirs)}")
print("=" * 60)



---

## Section 4 — Dataset Distribution Visualization


In [ ]:
"""
Section 4: Dataset Distribution Visualization
---------------------------------------------
Plots quantitative bar charts showing patient allocation across dataset splits.
"""

def plot_dataset_partition_layout(train_size: int, val_size: int, test_size: int) -> None:
    """Render a clean bar chart representing dataset split distribution."""
    fig, ax = plt.subplots(figsize=(8, 5))
    categories = ["Training Set", "Validation Set", "Test Set"]
    counts = [train_size, val_size, test_size]
    colors = ["#2b5c8f", "#d95f02", "#7570b3"]

    bars = ax.bar(categories, counts, color=colors, width=0.5, edgecolor="black", linewidth=1.2)
    ax.set_ylabel("Number of 3D Patient Volumes", fontsize=12, fontweight="bold")
    ax.set_title("BraTS 2020 T1-Only Dataset Partitioning", fontsize=14, fontweight="bold")
    ax.grid(axis="y", linestyle="--", alpha=0.7)

    for bar in bars:
        height = bar.get_height()
        ax.annotate(f"{height} vols\n({height*PipelineConfig.VOLUME_SLICES:,} slices)",
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 5), textcoords="offset points",
                    ha="center", va="bottom", fontsize=10, fontweight="bold")

    plt.tight_layout()
    plt.show()


plot_dataset_partition_layout(len(train_patient_dirs), len(val_patient_dirs), len(test_patient_dirs))



---

## Section 5 — Qualitative NIfTI Volume & Slice Exploration


In [ ]:
"""
Section 5: Qualitative NIfTI Volume & Slice Exploration
-------------------------------------------------------
Visualizes raw T1 axial slices along with tumor sub-region mask overlays.

[v2 FIX] `get_fdata()` now explicitly requests `dtype=np.float32` instead of silently
defaulting to `float64`, halving memory bandwidth for this exploratory load.
"""

def inspect_patient_volume(patient_dir: str, slice_indices: List[int]) -> None:
    """
    Load 3D T1 and segmentation NIfTI files and plot multi-slice axial progression.

    Args:
        patient_dir (str): Path to patient folder.
        slice_indices (List[int]): Axial slice indices to render.
    """
    p_name = os.path.basename(patient_dir)
    t1_file = os.path.join(patient_dir, f"{p_name}_{PipelineConfig.MODALITY}.nii")
    seg_file = os.path.join(patient_dir, f"{p_name}_{PipelineConfig.TARGET_SUFFIX}.nii")

    t1_vol = nib.load(t1_file).get_fdata(dtype=np.float32)
    seg_vol = nib.load(seg_file).get_fdata(dtype=np.float32)

    num_cols = len(slice_indices)
    fig, axes = plt.subplots(2, num_cols, figsize=(4 * num_cols, 8))

    class_labels = {1: "Necrotic/Core (Red)", 2: "Edema (Green)", 4: "Enhancing (Yellow)"}

    for idx, s_idx in enumerate(slice_indices):
        t1_slice = np.rot90(t1_vol[:, :, s_idx])
        seg_slice = np.rot90(seg_vol[:, :, s_idx])

        # Top Row: Raw T1 Scan
        axes[0, idx].imshow(t1_slice, cmap="gray")
        axes[0, idx].set_title(f"T1 Slice #{s_idx}", fontsize=11, fontweight="bold")
        axes[0, idx].axis("off")

        # Bottom Row: T1 with Multi-Class Mask Overlay
        axes[1, idx].imshow(t1_slice, cmap="gray")

        # Build colored overlay masks
        overlay = np.zeros((*t1_slice.shape, 4))
        overlay[seg_slice == 1] = [1.0, 0.0, 0.0, 0.6]  # Red for Necrotic Core
        overlay[seg_slice == 2] = [0.0, 1.0, 0.0, 0.6]  # Green for Edema
        overlay[seg_slice == 4] = [1.0, 0.8, 0.0, 0.7]  # Yellow for Enhancing Tumor

        axes[1, idx].imshow(overlay)
        axes[1, idx].set_title(f"Segmentation Overlay #{s_idx}", fontsize=11)
        axes[1, idx].axis("off")

    plt.suptitle(f"Patient Exploration: {p_name} (Strictly T1 Modality)", fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.show()


# Display representative slices from the first training patient volume
sample_patient = train_patient_dirs[0]
sample_slices = [40, 60, 80, 100]
inspect_patient_volume(sample_patient, sample_slices)



---

## Section 6 — One-Time Slice Pre-Caching to `.npy` [v2 NEW]


In [ ]:
"""
Section 6: One-Time Slice Pre-Caching (.npy) -- [v2 NEW]
-----------------------------------------------------------
This is the fix for the #1 bottleneck identified in the training analysis:
`BraTS2DT1Dataset.__getitem__` previously reloaded and decompressed the *entire* 155-slice,
float64 3D NIfTI volume (~22MB) on every single slice access. With BATCH_SIZE=32, that meant
32 full volume loads discarded down to 32 slices, per training step.

This cell runs ONCE per patient set and converts every relevant axial slice (for train + val +
test patients) into a small float32 `.npy` file. After this, Section 7's `__getitem__` becomes
a near-free `np.load()` call. The function is idempotent -- already-cached slices are skipped on
re-run, so it is safe to re-execute this cell across sessions.
"""
import numpy as np
import nibabel as nib


def precache_patient_slices(patient_dirs: List[str], config) -> None:
    """Convert each patient's relevant axial slices to cached float32 .npy files."""
    os.makedirs(config.CACHE_DIR, exist_ok=True)

    for p_dir in tqdm(patient_dirs, desc="Pre-caching slices to .npy"):
        p_name = os.path.basename(p_dir)
        t1_path = os.path.join(p_dir, f"{p_name}_{config.MODALITY}.nii")
        seg_path = os.path.join(p_dir, f"{p_name}_{config.TARGET_SUFFIX}.nii")

        # Skip this patient entirely if the full slice range is already cached
        first_marker = os.path.join(config.CACHE_DIR, f"{p_name}_t1_s{config.VOLUME_START_AT:03d}.npy")
        last_marker = os.path.join(
            config.CACHE_DIR,
            f"{p_name}_t1_s{config.VOLUME_START_AT + config.VOLUME_SLICES - 1:03d}.npy"
        )
        if os.path.exists(first_marker) and os.path.exists(last_marker):
            continue

        try:
            # [v2 FIX] Explicit float32 dtype -- halves memory bandwidth vs. the float64 default
            t1_vol = nib.load(t1_path).get_fdata(dtype=np.float32)
            seg_vol = nib.load(seg_path).get_fdata(dtype=np.float32).astype(np.uint8)
        except Exception as e:
            print(f"[Pre-cache] Skipping {p_name} due to read failure: {e}")
            continue

        seg_vol[seg_vol == 4] = 3  # Re-map raw label 4 -> contiguous class index 3 once, up front

        for s_idx in range(config.VOLUME_START_AT, config.VOLUME_START_AT + config.VOLUME_SLICES):
            t1_out = os.path.join(config.CACHE_DIR, f"{p_name}_t1_s{s_idx:03d}.npy")
            seg_out = os.path.join(config.CACHE_DIR, f"{p_name}_seg_s{s_idx:03d}.npy")
            if not os.path.exists(t1_out):
                np.save(t1_out, t1_vol[:, :, s_idx])
            if not os.path.exists(seg_out):
                np.save(seg_out, seg_vol[:, :, s_idx])


precache_patient_slices(train_patient_dirs + val_patient_dirs + test_patient_dirs, PipelineConfig)
print(f"[Pre-cache] Slice cache ready at: {PipelineConfig.CACHE_DIR}")



---

## Section 7 — Robust Preprocessing Pipeline & PyTorch Dataset [v2 REWRITTEN]


In [ ]:
"""
Section 7: Robust Preprocessing Pipeline & PyTorch Dataset -- [v2 REWRITTEN]
------------------------------------------------------------------------------
Implements standardized intensity normalization (1st-99th percentile clipping + z-score),
high-speed spatial resizing, clean one-hot label encoding, and cache-backed indexing.

[v2 FIX] `__getitem__` now reads pre-cached per-slice `.npy` files (Section 6) instead of
reloading and decompressing the full 3D NIfTI volume on every access -- the single biggest
throughput fix in this notebook (10-50x faster DataLoader per the bottleneck analysis).
[v2 FIX] `NUM_WORKERS` from `PipelineConfig` is now actually passed to the DataLoaders.
It was defined but hardcoded to `0` in v1, silently disabling the configured parallel workers.
Because slice loading is now trivial `np.load()` I/O, multiple workers genuinely overlap I/O
with GPU compute instead of contending for slow full-volume disk reads.
"""
import os
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from typing import List, Tuple, Dict

class TensorPreprocessing:
    """Static helper methods for clinical MRI tensor transformations."""

    @staticmethod
    def normalize_intensity(slice_img: np.ndarray) -> np.ndarray:
        """
        Robust intensity normalization:
        Clip background noise at 1st percentile and hyper-intense artifacts at 99th percentile,
        followed by zero-mean unit-variance z-score normalization on non-zero tissue voxels.
        """
        mask = slice_img > 0
        if not np.any(mask):
            return np.zeros_like(slice_img, dtype=np.float32)

        p1, p99 = np.percentile(slice_img[mask], (1, 99))
        clipped = np.clip(slice_img, p1, p99)

        mean_val = np.mean(clipped[mask])
        std_val = np.std(clipped[mask]) + 1e-8

        normalized = np.zeros_like(slice_img, dtype=np.float32)
        normalized[mask] = (clipped[mask] - mean_val) / std_val
        return normalized

    @staticmethod
    def resize_slice(slice_img: np.ndarray, target_size: int, is_mask: bool = False) -> np.ndarray:
        """Resize 2D slice using bilinear interpolation for continuous MRI and nearest-neighbor for masks."""
        interpolation = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
        resized = cv2.resize(slice_img, (target_size, target_size), interpolation=interpolation)
        return resized


class BraTS2DT1Dataset(Dataset):
    """
    PyTorch Dataset optimized for 2D axial slice segmentation.
    Reads slices from the pre-cached .npy store built in Section 6 -- O(1) disk I/O per item.
    """
    def __init__(self, patient_dirs: List[str], config, augment: bool = False):
        super().__init__()
        self.patient_dirs = patient_dirs
        self.config = config
        self.augment = augment

        # Build linear index table mapping slice IDs to (patient_name, slice_index),
        # skipping empty (all-background) slices. Reads the tiny cached seg .npy files
        # instead of the full 3D volume, so building the index is now fast too.
        self.slice_registry: List[Tuple[str, int]] = []

        for p_dir in self.patient_dirs:
            p_name = os.path.basename(p_dir)
            for s_idx in range(
                self.config.VOLUME_START_AT,
                self.config.VOLUME_START_AT + self.config.VOLUME_SLICES
            ):
                seg_cache_path = os.path.join(self.config.CACHE_DIR, f"{p_name}_seg_s{s_idx:03d}.npy")
                seg_slice = np.load(seg_cache_path)
                if np.any(seg_slice > 0):
                    self.slice_registry.append((p_name, s_idx))

    def __len__(self) -> int:
        return len(self.slice_registry)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        p_name, s_idx = self.slice_registry[idx]

        # [v2 FIX] Trivial np.load() instead of nib.load(...).get_fdata() on the full volume
        raw_t1 = np.load(os.path.join(self.config.CACHE_DIR, f"{p_name}_t1_s{s_idx:03d}.npy")).astype(np.float32)
        raw_seg = np.load(os.path.join(self.config.CACHE_DIR, f"{p_name}_seg_s{s_idx:03d}.npy")).astype(np.uint8)

        # Apply intensity normalization and resizing
        norm_t1 = TensorPreprocessing.normalize_intensity(raw_t1)
        res_t1 = TensorPreprocessing.resize_slice(norm_t1, self.config.IMG_SIZE, is_mask=False)
        res_seg = TensorPreprocessing.resize_slice(raw_seg, self.config.IMG_SIZE, is_mask=True)

        # Label 4 -> 3 remap already applied during pre-caching; kept here as a defensive no-op.
        res_seg[res_seg == 4] = 3

        # Optional lightweight spatial augmentation (Random horizontal/vertical flips)
        if self.augment:
            if random.random() > 0.5:
                res_t1 = np.fliplr(res_t1)
                res_seg = np.fliplr(res_seg)
            if random.random() > 0.5:
                res_t1 = np.flipud(res_t1)
                res_seg = np.flipud(res_seg)

        # Convert to PyTorch tensors (Shape: [Channels, H, W])
        tensor_t1 = torch.from_numpy(res_t1.copy()).unsqueeze(0).float()
        tensor_seg = torch.from_numpy(res_seg.copy()).long()

        return tensor_t1, tensor_seg


train_dataset = BraTS2DT1Dataset(
    train_patient_dirs,
    PipelineConfig,
    augment=True
)

val_dataset = BraTS2DT1Dataset(
    val_patient_dirs,
    PipelineConfig,
    augment=False
)

test_dataset = BraTS2DT1Dataset(
    test_patient_dirs,
    PipelineConfig,
    augment=False
)

# [v2 FIX] NUM_WORKERS now comes from PipelineConfig instead of being hardcoded to 0.
# persistent_workers avoids re-spawning worker processes every epoch.
train_loader = DataLoader(
    train_dataset,
    batch_size=PipelineConfig.BATCH_SIZE,
    shuffle=True,
    num_workers=PipelineConfig.NUM_WORKERS,
    pin_memory=PipelineConfig.PIN_MEMORY,
    persistent_workers=PipelineConfig.NUM_WORKERS > 0,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=PipelineConfig.BATCH_SIZE,
    shuffle=False,
    num_workers=PipelineConfig.NUM_WORKERS,
    pin_memory=PipelineConfig.PIN_MEMORY,
    persistent_workers=PipelineConfig.NUM_WORKERS > 0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=PipelineConfig.BATCH_SIZE,
    shuffle=False,
    num_workers=PipelineConfig.NUM_WORKERS,
    pin_memory=PipelineConfig.PIN_MEMORY,
    persistent_workers=PipelineConfig.NUM_WORKERS > 0
)

print(f"Training slices   : {len(train_dataset)}")
print(f"Validation slices : {len(val_dataset)}")
print(f"Testing slices    : {len(test_dataset)}")

print(f"Training batches  : {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")



---

## Section 8 — Lightweight Attention U-Net Architecture


In [ ]:
"""
Section 8: Lightweight Attention U-Net Architecture Optimized for Dual Tesla T4
-------------------------------------------------------------------------------
Features modular convolution blocks, spatial attention gates to focus on tumor margins,
Mish activations, Batch Normalization, Kaiming weight initialization, and multi-GPU DataParallel scaling.
"""

class ConvBlock(nn.Module):
    """Double 3x3 Convolution layer with BatchNorm and Mish activations."""
    def __init__(self, in_channels: int, out_channels: int, dropout_rate: float = 0.1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.Mish(inplace=True),
            nn.Dropout2d(dropout_rate),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.Mish(inplace=True)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class AttentionGate(nn.Module):
    """Spatial Attention Gate to filter skip connection activations before merging."""
    def __init__(self, f_g: int, f_l: int, f_int: int):
        super().__init__()
        self.w_g = nn.Sequential(
            nn.Conv2d(f_g, f_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(f_int)
        )
        self.w_x = nn.Sequential(
            nn.Conv2d(f_l, f_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(f_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(f_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        g1 = self.w_g(g)
        x1 = self.w_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi


class AttentionUNet(nn.Module):
    """
    Lightweight Attention U-Net architecture designed for 1-channel T1 MRI segmentation
    into 4 multi-class tumor sub-regions.
    """
    def __init__(self, in_channels: int = 1, num_classes: int = 4):
        super().__init__()

        # Encoder Down-sampling stage
        self.conv1 = ConvBlock(in_channels, 32)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = ConvBlock(32, 64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv3 = ConvBlock(64, 128)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv4 = ConvBlock(128, 256)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Bottleneck
        self.bottleneck = ConvBlock(256, 512, dropout_rate=0.3)

        # Decoder Up-sampling with Attention Gates
        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.att4 = AttentionGate(f_g=256, f_l=256, f_int=128)
        self.dec4 = ConvBlock(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.att3 = AttentionGate(f_g=128, f_l=128, f_int=64)
        self.dec3 = ConvBlock(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.att2 = AttentionGate(f_g=64, f_l=64, f_int=32)
        self.dec2 = ConvBlock(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.att1 = AttentionGate(f_g=32, f_l=32, f_int=16)
        self.dec1 = ConvBlock(64, 32)

        # Final Segmentation Output Projection
        self.final_conv = nn.Conv2d(32, num_classes, kernel_size=1)

        self._initialize_weights()

    def _initialize_weights(self) -> None:
        """Apply Kaiming Normal weight initialization across all convolutional layers."""
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Encoding
        x1 = self.conv1(x)
        x2 = self.conv2(self.pool1(x1))
        x3 = self.conv3(self.pool2(x2))
        x4 = self.conv4(self.pool3(x3))

        # Bottleneck
        b = self.bottleneck(self.pool4(x4))

        # Decoding with Attention Gates
        d4 = self.up4(b)
        x4_att = self.att4(g=d4, x=x4)
        d4 = self.dec4(torch.cat([x4_att, d4], dim=1))

        d3 = self.up3(d4)
        x3_att = self.att3(g=d3, x=x3)
        d3 = self.dec3(torch.cat([x3_att, d3], dim=1))

        d2 = self.up2(d3)
        x2_att = self.att2(g=d2, x=x2)
        d2 = self.dec2(torch.cat([x2_att, d2], dim=1))

        d1 = self.up1(d2)
        x1_att = self.att1(g=d1, x=x1)
        d1 = self.dec1(torch.cat([x1_att, d1], dim=1))

        logits = self.final_conv(d1)
        return logits


def instantiate_production_model() -> nn.Module:
    """Build model and wrap in DataParallel if dual GPUs are available."""
    model = AttentionUNet(in_channels=1, num_classes=PipelineConfig.NUM_CLASSES)
    model = model.to(DEVICE)

    if torch.cuda.device_count() > 1:
        print(f"[Parallel Scaling] Enabling multi-GPU DataParallel across {torch.cuda.device_count()} GPUs.")
        model = nn.DataParallel(model)

    return model


model = instantiate_production_model()
print(f"[Model Verification] Total Trainable Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")



---

## Section 9 — Combined Loss Functions & Clinical Validation Metrics [v3 REBALANCED]


In [ ]:
"""
Section 9: Combined Loss Functions & Clinical Validation Metrics -- [v3 REBALANCED]
--------------------------------------------------------------------------------------
Combines Cross-Entropy Loss with a Multi-Class Tversky Loss to address severe class imbalance.
Computes Dice, IoU, Precision, Recall, F1, and Hausdorff Distance 95 (HD95).

[v3 FIX] `MultiClassDiceLoss` replaced with `MultiClassTverskyLoss`. Plain Dice weighs false
negatives and false positives equally, which under-penalizes the small, easily-missed
structures (ET, NCR) relative to the dominant background/ED classes -- consistent with the
v2 training run where ET Dice stayed stuck near 0.25-0.28 while ED reached 0.45-0.47.
Tversky's `alpha` (false-negative weight) is set above `beta` (false-positive weight) so the
loss penalizes *missed* tumor pixels more than over-segmentation.
[v3 FIX] `CombinedSegmentationLoss` weighting shifted from 0.5/0.5 to
`PipelineConfig.CE_WEIGHT` / `PipelineConfig.TVERSKY_WEIGHT` (0.3/0.7 by default) -- per-pixel
CE is dominated by the easy, overwhelmingly common background class, so down-weighting it lets
the segmentation-aware Tversky term drive more of the gradient.
"""

class MultiClassTverskyLoss(nn.Module):
    """
    Tversky loss computed across individual tumor classes (excluding background).
    `alpha` weights false negatives, `beta` weights false positives. Setting alpha > beta
    (the v3 default: 0.7 / 0.3) makes the loss punish missed tumor pixels more than
    over-segmented ones -- useful when small structures like ET are chronically under-detected.
    Reduces to standard Dice loss when alpha == beta == 0.5.
    """
    def __init__(self, alpha: float = 0.7, beta: float = 0.3, smooth: float = 1e-6):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.smooth = smooth

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = F.softmax(logits, dim=1)
        targets_one_hot = F.one_hot(targets, num_classes=PipelineConfig.NUM_CLASSES).permute(0, 3, 1, 2).float()

        total_loss = 0.0
        # Compute loss primarily over active tumor classes (indices 1, 2, 3)
        for cls_idx in range(1, PipelineConfig.NUM_CLASSES):
            p_flat = probs[:, cls_idx].contiguous().view(-1)
            t_flat = targets_one_hot[:, cls_idx].contiguous().view(-1)

            true_positive = (p_flat * t_flat).sum()
            false_negative = ((1.0 - p_flat) * t_flat).sum()
            false_positive = (p_flat * (1.0 - t_flat)).sum()

            tversky_score = (true_positive + self.smooth) / (
                true_positive + self.alpha * false_negative + self.beta * false_positive + self.smooth
            )
            total_loss += (1.0 - tversky_score)

        return total_loss / (PipelineConfig.NUM_CLASSES - 1)


class CombinedSegmentationLoss(nn.Module):
    """Weighted combination of CrossEntropyLoss and MultiClassTverskyLoss."""
    def __init__(
        self,
        ce_weight: float = PipelineConfig.CE_WEIGHT,
        tversky_weight: float = PipelineConfig.TVERSKY_WEIGHT,
        tversky_alpha: float = PipelineConfig.TVERSKY_ALPHA,
        tversky_beta: float = PipelineConfig.TVERSKY_BETA,
    ):
        super().__init__()
        self.ce_loss = nn.CrossEntropyLoss()
        self.tversky_loss = MultiClassTverskyLoss(alpha=tversky_alpha, beta=tversky_beta)
        self.ce_weight = ce_weight
        self.tversky_weight = tversky_weight

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        loss_ce = self.ce_loss(logits, targets)
        loss_tversky = self.tversky_loss(logits, targets)
        return self.ce_weight * loss_ce + self.tversky_weight * loss_tversky


class ClinicalMetrics:
    """Evaluates quantitative clinical segmentation metrics per batch."""

    @staticmethod
    def compute_dice_per_class(preds: torch.Tensor, targets: torch.Tensor, smooth: float = 1e-6) -> Dict[str, float]:
        """Calculate exact Dice score for NCR/NET, Edema, and Enhancing tumor."""
        metrics = {}
        class_names = {1: "Dice_Necrotic", 2: "Dice_Edema", 3: "Dice_Enhancing"}

        for cls_idx, name in class_names.items():
            p_bin = (preds == cls_idx).float().view(-1)
            t_bin = (targets == cls_idx).float().view(-1)
            intersection = (p_bin * t_bin).sum().item()
            union = p_bin.sum().item() + t_bin.sum().item()
            metrics[name] = (2.0 * intersection + smooth) / (union + smooth) if union > 0 else 1.0

        metrics["Dice_Mean"] = sum(metrics.values()) / len(metrics)
        return metrics

    @staticmethod
    def compute_iou_precision_recall(preds: torch.Tensor, targets: torch.Tensor, smooth: float = 1e-6) -> Dict[str, float]:
        """Calculate IoU, Precision, Recall, and F1 across foreground tumor regions."""
        p_fg = (preds > 0).float().view(-1)
        t_fg = (targets > 0).float().view(-1)

        tp = (p_fg * t_fg).sum().item()
        fp = (p_fg * (1.0 - t_fg)).sum().item()
        fn = ((1.0 - p_fg) * t_fg).sum().item()

        iou = (tp + smooth) / (tp + fp + fn + smooth)
        precision = (tp + smooth) / (tp + fp + smooth)
        recall = (tp + smooth) / (tp + fn + smooth)
        f1 = (2.0 * precision * recall) / (precision + recall + smooth)

        return {"IoU": iou, "Precision": precision, "Recall": recall, "F1": f1}

    @staticmethod
    def compute_hd95_approx(preds: np.ndarray, targets: np.ndarray) -> float:
        """Compute 95th percentile Hausdorff Distance using exact Euclidean distance transforms."""
        p_fg = preds > 0
        t_fg = targets > 0

        if not np.any(p_fg) or not np.any(t_fg):
            return 0.0 if not np.any(p_fg) and not np.any(t_fg) else 50.0

        dt_p = ndimage.distance_transform_edt(~p_fg)
        dt_t = ndimage.distance_transform_edt(~t_fg)

        dist_p_to_t = dt_t[p_fg]
        dist_t_to_p = dt_p[t_fg]

        hd95_val = max(np.percentile(dist_p_to_t, 95), np.percentile(dist_t_to_p, 95))
        return float(hd95_val)


criterion = CombinedSegmentationLoss()



---

## Section 10 — Complete Production Training Engine with AMP & Checkpointing [v3 REVISED]


In [ ]:
"""
Section 10: Complete Production Training Engine with AMP & Checkpointing -- [v3 REVISED]
-------------------------------------------------------------------------------------------
Encapsulates optimizer, plateau-based LR scheduler, mixed precision GradScaler,
checkpoint saving/resuming, and TensorBoard metric logging.

[v3 FIX] Scheduler switched from `CosineAnnealingLR` to `ReduceLROnPlateau`, monitored on
val Dice (`mode="max"`). The v2 run showed val Dice oscillating epoch-to-epoch from epoch 9
onward (0.3407 -> 0.3367 -> 0.3153 -> 0.3214) while train loss kept dropping smoothly --
the LR was still too high for that stage under a fixed cosine schedule. `ReduceLROnPlateau`
only decays the LR (by `LR_PLATEAU_FACTOR`) once val Dice actually stalls for
`LR_PLATEAU_PATIENCE` epochs, instead of decaying on a fixed epoch count regardless of
whether the model is still improving.
"""

class BraTSTrainer:
    def __init__(self, model: nn.Module, criterion: nn.Module, config: PipelineConfig):
        self.model = model
        self.criterion = criterion
        self.config = config

        self.optimizer = torch.optim.AdamW(
            self.model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY
        )

        # [v3 FIX] Plateau-based schedule -- decays LR only when val Dice stops improving,
        # rather than on a fixed cosine curve regardless of actual progress.
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer,
            mode="max",
            factor=config.LR_PLATEAU_FACTOR,
            patience=config.LR_PLATEAU_PATIENCE,
            min_lr=config.LR_MIN,
        )
        self.scaler = GradScaler(enabled=config.USE_AMP)
        self.writer = SummaryWriter(log_dir=config.LOG_DIR)

        self.best_val_dice = 0.0
        self.start_epoch = 1

    def train_epoch(self, dataloader: DataLoader, epoch: int) -> float:
        self.model.train()
        total_loss = 0.0

        progress = tqdm(dataloader, desc=f"Epoch {epoch:02d}/{self.config.EPOCHS} [Train]", leave=False)
        for images, targets in progress:
            images, targets = images.to(DEVICE, non_blocking=True), targets.to(DEVICE, non_blocking=True)
            self.optimizer.zero_grad(set_to_none=True)

            with autocast(enabled=self.config.USE_AMP):
                logits = self.model(images)
                loss = self.criterion(logits, targets)

            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.GRAD_CLIP_NORM)

            self.scaler.step(self.optimizer)
            self.scaler.update()

            total_loss += loss.item()
            progress.set_postfix({"Loss": f"{loss.item():.4f}"})

        return total_loss / len(dataloader)

    @torch.no_grad()
    def validate_epoch(self, dataloader: DataLoader, epoch: int) -> Dict[str, float]:
        self.model.eval()
        total_loss = 0.0
        metrics_sum = {"Dice_Necrotic": 0.0, "Dice_Edema": 0.0, "Dice_Enhancing": 0.0, "Dice_Mean": 0.0, "IoU": 0.0}

        progress = tqdm(dataloader, desc=f"Epoch {epoch:02d}/{self.config.EPOCHS} [Val]", leave=False)
        for images, targets in progress:
            images, targets = images.to(DEVICE, non_blocking=True), targets.to(DEVICE, non_blocking=True)

            with autocast(enabled=self.config.USE_AMP):
                logits = self.model(images)
                loss = self.criterion(logits, targets)

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)

            batch_dice = ClinicalMetrics.compute_dice_per_class(preds, targets)
            batch_iou = ClinicalMetrics.compute_iou_precision_recall(preds, targets)["IoU"]

            for k in batch_dice:
                metrics_sum[k] += batch_dice[k]
            metrics_sum["IoU"] += batch_iou

        num_batches = len(dataloader)
        avg_metrics = {k: v / num_batches for k, v in metrics_sum.items()}
        avg_metrics["Val_Loss"] = total_loss / num_batches
        return avg_metrics

    def save_checkpoint(self, epoch: int, metrics: Dict[str, float], is_best: bool) -> None:
        state = {
            "epoch": epoch,
            "model_state_dict": self.model.module.state_dict() if isinstance(self.model, nn.DataParallel) else self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "best_val_dice": self.best_val_dice,
            "metrics": metrics
        }
        ckpt_path = os.path.join(self.config.CHECKPOINT_DIR, "latest_checkpoint.pth")
        torch.save(state, ckpt_path)

        if is_best:
            best_path = os.path.join(self.config.CHECKPOINT_DIR, "best_t1_segmentation_model.pth")
            shutil.copy(ckpt_path, best_path)
            print(f"  [Checkpoint] New optimal model saved! Mean Dice: {metrics['Dice_Mean']:.4f}")

    def fit(self, train_loader: DataLoader, val_loader: DataLoader) -> List[Dict[str, float]]:
        history = []
        print("\n" + "="*70)
        print("STARTING PRODUCTION TRAINING LOOP (STRICT T1 MODALITY)")
        print("="*70)

        for epoch in range(self.start_epoch, self.config.EPOCHS + 1):
            train_loss = self.train_epoch(train_loader, epoch)
            val_metrics = self.validate_epoch(val_loader, epoch)
            val_dice = val_metrics["Dice_Mean"]

            # [v3 FIX] ReduceLROnPlateau needs the monitored metric passed to .step()
            self.scheduler.step(val_dice)

            is_best = val_dice > self.best_val_dice
            if is_best:
                self.best_val_dice = val_dice

            self.save_checkpoint(epoch, val_metrics, is_best)

            # Log metrics to TensorBoard
            self.writer.add_scalar("Loss/Train", train_loss, epoch)
            self.writer.add_scalar("Loss/Val", val_metrics["Val_Loss"], epoch)
            self.writer.add_scalar("Dice/Mean_Val", val_dice, epoch)
            self.writer.add_scalar("LR", self.optimizer.param_groups[0]["lr"], epoch)

            log_entry = {
                "Epoch": epoch, "Train_Loss": train_loss, "Val_Loss": val_metrics["Val_Loss"],
                "Val_Dice_Mean": val_dice, "Val_IoU": val_metrics["IoU"]
            }
            history.append(log_entry)

            print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_metrics['Val_Loss']:.4f} | "
                  f"Val Dice: {val_dice:.4f} (NCR:{val_metrics['Dice_Necrotic']:.2f}, ED:{val_metrics['Dice_Edema']:.2f}, ET:{val_metrics['Dice_Enhancing']:.2f})")

        self.writer.close()
        print("="*70)
        print(f"TRAINING COMPLETED. Optimal Validation Mean Dice Score: {self.best_val_dice:.4f}")
        return history


trainer = BraTSTrainer(model, criterion, PipelineConfig)



---

## Section 11 — Pre-Flight System Check [v2 CONSOLIDATED]


In [ ]:
"""
Section 11: Pre-Flight System Check -- [v2 CONSOLIDATED]
------------------------------------------------------------
[v2 FIX] v1 scattered these sanity checks (CUDA availability, GPU name, model device,
batch size, image size, parameter count) across three separate leftover debug cells.
Consolidated into a single quick pre-flight check before kicking off training.
"""

print(f"CUDA available     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU                 : {torch.cuda.get_device_name(0)}")
print(f"Model device        : {next(model.parameters()).device}")
print(f"Batch size          : {PipelineConfig.BATCH_SIZE}")
print(f"Image size          : {PipelineConfig.IMG_SIZE}")
print(f"DataLoader workers  : {PipelineConfig.NUM_WORKERS}")

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters    : {total_params:,}")



---

## Section 12 — Model Execution & Convergence Curve Rendering


In [ ]:
"""
Section 12: Model Execution & Convergence Curve Rendering
---------------------------------------------------------
Executes the training schedule and plots learning progression across epochs.
"""

# Execute training pipeline (Using a reduced 5-epoch demonstration loop if running on mock CPU environment)
effective_epochs = 5 if DEVICE.type == "cpu" else PipelineConfig.EPOCHS
trainer.config.EPOCHS = effective_epochs

training_history = trainer.fit(train_loader, val_loader)
history_df = pd.DataFrame(training_history)

def plot_training_curves(df: pd.DataFrame) -> None:
    """Render loss trajectory and Dice coefficient convergence charts."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left Chart: Loss Progression
    axes[0].plot(df["Epoch"], df["Train_Loss"], label="Training Loss", color="#1f77b4", marker="o", linewidth=2)
    axes[0].plot(df["Epoch"], df["Val_Loss"], label="Validation Loss", color="#d62728", marker="s", linewidth=2)
    axes[0].set_xlabel("Epoch", fontweight="bold")
    axes[0].set_ylabel("Combined BCE + Dice Loss", fontweight="bold")
    axes[0].set_title("Training & Validation Loss Trajectory", fontsize=13, fontweight="bold")
    axes[0].legend()
    axes[0].grid(True, linestyle="--", alpha=0.6)

    # Right Chart: Mean Dice Convergence
    axes[1].plot(df["Epoch"], df["Val_Dice_Mean"], label="Validation Mean Dice", color="#2ca02c", marker="^", linewidth=2.5)
    axes[1].plot(df["Epoch"], df["Val_IoU"], label="Validation IoU", color="#9467bd", marker="d", linewidth=2)
    axes[1].set_xlabel("Epoch", fontweight="bold")
    axes[1].set_ylabel("Score", fontweight="bold")
    axes[1].set_title("Validation Segmentation Accuracy Progression", fontsize=13, fontweight="bold")
    axes[1].legend()
    axes[1].grid(True, linestyle="--", alpha=0.6)

    plt.tight_layout()
    plt.show()


plot_training_curves(history_df)



---

## Section 13 — Comprehensive Quantitative Validation Benchmark


In [ ]:
"""
Section 13: Comprehensive Quantitative Validation Benchmark
-----------------------------------------------------------
Loads optimal saved weights and computes comprehensive clinical segmentation benchmarks
including Dice, IoU, Precision, Recall, F1, and Hausdorff Distance 95 over the validation set.
"""

def evaluate_production_benchmark(model: nn.Module, dataloader: DataLoader) -> pd.DataFrame:
    """Run full evaluation suite over validation slices and return summary report."""
    best_weights_path = os.path.join(PipelineConfig.CHECKPOINT_DIR, "best_t1_segmentation_model.pth")
    if os.path.exists(best_weights_path):
        checkpoint = torch.load(best_weights_path, map_location=DEVICE)
        target_model = model.module if isinstance(model, nn.DataParallel) else model
        target_model.load_state_dict(checkpoint["model_state_dict"])
        print(f"[Benchmark] Successfully loaded optimal checkpoint from Epoch {checkpoint['epoch']}")
    else:
        print("[Benchmark] Optimal checkpoint not found. Evaluating current model weights.")

    model.eval()
    accumulated_metrics = {
        "Dice_Necrotic": [], "Dice_Edema": [], "Dice_Enhancing": [],
        "Dice_Mean": [], "IoU": [], "Precision": [], "Recall": [], "F1": [], "HD95": []
    }

    with torch.no_grad():
        for images, targets in tqdm(dataloader, desc="Running Quantitative Benchmark"):
            images = images.to(DEVICE, non_blocking=True)
            targets_np = targets.numpy()

            logits = model(images)
            preds = torch.argmax(logits, dim=1)
            preds_np = preds.cpu().numpy()

            dice_dict = ClinicalMetrics.compute_dice_per_class(preds.cpu(), targets)
            gen_dict = ClinicalMetrics.compute_iou_precision_recall(preds.cpu(), targets)

            # Compute slice-wise HD95 distance
            hd95_val = ClinicalMetrics.compute_hd95_approx(preds_np[0], targets_np[0])

            for k, v in dice_dict.items():
                accumulated_metrics[k].append(v)
            for k, v in gen_dict.items():
                accumulated_metrics[k].append(v)
            accumulated_metrics["HD95"].append(hd95_val)

    summary_data = {}
    for k, v_list in accumulated_metrics.items():
        summary_data[k] = f"{np.mean(v_list):.4f} \u00b1 {np.std(v_list):.4f}"

    report_df = pd.DataFrame([summary_data], index=["T1_Attention_UNet_Evaluation"]).T
    report_df.columns = ["Score (Mean \u00b1 SD)"]
    return report_df


benchmark_report = evaluate_production_benchmark(model, val_loader)
print("\n" + "="*60)
print("CLINICAL VALIDATION REPORT (STRICT T1 MRI MODALITY)")
print("="*60)
print(benchmark_report)
print("="*60)



---

### Interpreting these numbers: recalibrated expectations for T1-only training [v3 NOTE]

> [!NOTE]
> **A mean Dice around 0.35–0.45 is a plausible practical ceiling for this T1-only setup — not evidence of a broken pipeline.**
>
> BraTS ground-truth sub-regions are each best defined by a specific modality:
> - **Enhancing Tumor (ET)** is, by definition, the region that lights up after gadolinium contrast on **T1CE**. Native (non-contrast) T1 does not contain this signal at all, so any ET Dice above near-zero reflects the model inferring enhancement indirectly from texture/intensity proxies — a fundamentally weaker cue.
> - **Peritumoral Edema (ED)** is primarily a **T2/FLAIR** hyperintensity finding; T1 shows it only faintly.
> - **Necrotic Core (NCR)** is the one sub-region reasonably visible on native T1, which is consistent with it tracking closer to ED than ET in practice.
>
> If your validation run shows the same pattern as above (ET Dice low and slow to rise, ED highest, NCR in between, mean Dice plateauing well below what full multi-modal BraTS pipelines report, typically 0.75–0.85 mean Dice), that is the expected signature of modality-restricted training, not a bug to keep chasing with hyperparameter tweaks.
>
> **For write-up / publication framing:** this is a legitimate contribution in its own right — a quantified feasibility study of tumor sub-region segmentation from native T1 alone, relevant to resource-constrained clinical settings where a full multi-modal MRI suite isn't always available (this framing already appears in the notebook's own clinical-context motivation). Report the per-class Dice breakdown (not just the mean) as your primary evidence: the ET/ED/NCR gap *is* the finding, showing which sub-regions remain recoverable without contrast-enhanced or T2/FLAIR sequences and which do not. Position the multi-modal literature numbers as the upper-bound reference point, and discuss the ET gap explicitly as the expected cost of the single-modality constraint rather than a limitation to apologize for.

---

## Section 14 — Production Inference Pipeline & Qualitative Visualizations


In [ ]:
"""
Section 14: Production Inference Pipeline & Qualitative Visualizations
----------------------------------------------------------------------
Runs end-to-end inference on test patient volumes, displaying side-by-side comparative
panels contrasting ground truth against model prediction masks.

[v2 FIX] `get_fdata()` now explicitly requests `dtype=np.float32` instead of silently
defaulting to `float64`.
"""

def generate_qualitative_inference_panel(model: nn.Module, patient_dir: str, slice_idx: int = 65) -> None:
    """
    Perform inference on a complete axial slice and display diagnostic multi-panel layout.
    """
    model.eval()
    p_name = os.path.basename(patient_dir)
    t1_path = os.path.join(patient_dir, f"{p_name}_{PipelineConfig.MODALITY}.nii")
    seg_path = os.path.join(patient_dir, f"{p_name}_{PipelineConfig.TARGET_SUFFIX}.nii")

    t1_vol = nib.load(t1_path).get_fdata(dtype=np.float32)
    seg_vol = nib.load(seg_path).get_fdata(dtype=np.float32)

    raw_t1 = t1_vol[:, :, slice_idx].astype(np.float32)
    raw_seg = seg_vol[:, :, slice_idx].astype(np.uint8)
    raw_seg[raw_seg == 4] = 3  # Normalize label 4 -> 3

    norm_t1 = TensorPreprocessing.normalize_intensity(raw_t1)
    res_t1 = TensorPreprocessing.resize_slice(norm_t1, PipelineConfig.IMG_SIZE, is_mask=False)
    res_seg = TensorPreprocessing.resize_slice(raw_seg, PipelineConfig.IMG_SIZE, is_mask=True)

    input_tensor = torch.from_numpy(res_t1).unsqueeze(0).unsqueeze(0).float().to(DEVICE)

    with torch.no_grad():
        logits = model(input_tensor)
        pred_mask = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()

    # Rotate 90 degrees for standard radiologic orientation
    rot_t1 = np.rot90(res_t1)
    rot_gt = np.rot90(res_seg)
    rot_pred = np.rot90(pred_mask)

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    # 1. Native T1 MRI Slice
    axes[0].imshow(rot_t1, cmap="gray")
    axes[0].set_title(f"Native T1 Scan (Slice #{slice_idx})", fontsize=12, fontweight="bold")
    axes[0].axis("off")

    # 2. Ground Truth Segmentation
    axes[1].imshow(rot_t1, cmap="gray")
    gt_overlay = np.zeros((*rot_t1.shape, 4))
    gt_overlay[rot_gt == 1] = [1.0, 0.0, 0.0, 0.6]
    gt_overlay[rot_gt == 2] = [0.0, 1.0, 0.0, 0.6]
    gt_overlay[rot_gt == 3] = [1.0, 0.8, 0.0, 0.7]
    axes[1].imshow(gt_overlay)
    axes[1].set_title("Ground Truth Mask Overlay", fontsize=12, fontweight="bold")
    axes[1].axis("off")

    # 3. Model Predicted Segmentation
    axes[2].imshow(rot_t1, cmap="gray")
    pred_overlay = np.zeros((*rot_t1.shape, 4))
    pred_overlay[rot_pred == 1] = [1.0, 0.0, 0.0, 0.6]
    pred_overlay[rot_pred == 2] = [0.0, 1.0, 0.0, 0.6]
    pred_overlay[rot_pred == 3] = [1.0, 0.8, 0.0, 0.7]
    axes[2].imshow(pred_overlay)
    axes[2].set_title("Attention U-Net Prediction Overlay", fontsize=12, fontweight="bold")
    axes[2].axis("off")

    # 4. Error Map / Difference Analysis
    diff_mask = (rot_gt != rot_pred).astype(np.float32)
    axes[3].imshow(rot_t1, cmap="gray")
    axes[3].imshow(diff_mask, cmap="Reds", alpha=0.4 * diff_mask)
    axes[3].set_title("Discrepancy Error Map (Red = Mismatch)", fontsize=12, fontweight="bold")
    axes[3].axis("off")

    plt.suptitle(f"Diagnostic Prediction Panel - Patient Case: {p_name}", fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.show()


# Run qualitative inference on sample unseen test patient volumes
for test_case in test_patient_dirs[:3]:
    generate_qualitative_inference_panel(model, test_case, slice_idx=60)



---

## Removed from v1 (cleanup, no functional loss)

The following v1 cells were leftover interactive debugging cruft and were removed in v2 — none of them contributed to the actual pipeline:

- A cell re-scanning + re-splitting the dataset a second time (redundant with Section 3).
- A cell printing `os.path.exists(PipelineConfig.TRAIN_DATASET_DIR)` and listing the first 3 patient folders.
- A cell listing `/kaggle/input` contents.
- A cell doing an `os.walk` search for `synthetic_brats_data`.
- Three separate one-line `print()` debug cells (`torch.cuda.is_available()` / device checks, `IMG_SIZE`, parameter count) — merged into the single Section 11 pre-flight check.
